# Code-first training (local, offline)

Train and compare code-first candidates with time-aware validation, then
select a champion. **All data is synthetic.** No Azure resources are used.

Prerequisite: `uv sync --extra notebooks` (or `--all-extras`).

In [ ]:
from revenue_prediction.config.loader import load_settings
from revenue_prediction.data.synthetic import generate_synthetic_dataset
from revenue_prediction.data.contracts import validate_raw_snapshots, validate_leakage_rules

settings = load_settings('dev')
df = generate_synthetic_dataset(settings.data)
validate_raw_snapshots(df)
validate_leakage_rules(df)
df.head()

In [ ]:
from revenue_prediction.training.splitting import blocked_temporal_split
from revenue_prediction.training.train import train_all_candidates
from revenue_prediction.evaluation.selection import select_champion_challenger

split = blocked_temporal_split(df, settings.split)
results = train_all_candidates(split, settings.model, settings.features)
selection = select_champion_challenger(results, settings.evaluation, metric=settings.model.primary_metric)
selection.ranking

In [ ]:
champ = results[selection.champion]
print('Champion:', selection.champion)
print('\nAccuracy by snapshot day:')
print(champ.by_snapshot_day.to_string(index=False))
print('\nAccuracy by facility:')
print(champ.by_facility.to_string(index=False))

## Submitting to Azure ML (optional)

With the `azure` extra installed and a configured workspace, build and submit a
command job. This cell is illustrative; it requires real credentials.

In [ ]:
# from revenue_prediction.azureml.client import get_ml_client
# from revenue_prediction.azureml.jobs import build_command_job
# ml_client = get_ml_client(settings.azure_ml)
# job = build_command_job(settings.azure_ml, training_data_asset='azureml:revenue_snapshots@latest',
#                         environment='azureml:revenue-prediction-env@latest')
# submitted = ml_client.jobs.create_or_update(job)
# print(submitted.studio_url)